# Notebook 01 — Data Preparation & Cleaning
### CREA Anomaly Detection · PM2.5 Pipeline

**Objective:** transform the raw inputs (`anomaly_locations.csv` +
`anomaly_measurements.csv`) into a single analysis-ready cleaned dataset
(`anomaly_merged.csv`).

The subsequent notebook (`02_exploratory_data_analysis.ipynb`) reads
`anomaly_merged.csv` as its starting point, so NB01 must be run to completion
before opening NB02.

## 📋 Table of Contents

| Section | Content |
|---|---|
| **Setup** | Load libraries, establish centralized configuration (`params.yml`) |
| | Load raw data, validate columns |
| **A** | Clean `locations`: structure, duplicates, coordinates, country standardization |
| **B** | Clean `measurements`: missingness, pollutant/unit, duplicates, sentinels, negatives, datetime, cleaning log |
| **C** | Merge `locations` + `measurements`, merge quality check |
| **D** | Cleaning summary, descriptive statistics (log-normal-aware) |

## 1. Load Libraries & Centralized Configuration

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yaml
from pathlib import Path

CONFIG_PATH = Path("../config/params.yml")

try:
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        CONFIG = yaml.safe_load(f)
    print(f"✅ Configuration loaded from {CONFIG_PATH}")
    print(f"   config_version: {CONFIG['meta']['config_version']}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"❌ {CONFIG_PATH} not found. This file is the source for all "
        f"parameters — the notebook must not run without it. "
        f"Verify the repository structure: crea-anomaly-detection/config/params.yml"
    )

required_sections = ["meta", "cleaning", "eda", "derivation", "filter", "spatial", "tsad", "paths", "colors_map"]
missing = [s for s in required_sections if s not in CONFIG]
assert not missing, f"❌ Missing section(s) in params.yml: {missing}"

# expected_countries is read from YAML as a list -> convert to a set for set operations
CONFIG["cleaning"]["expected_countries"] = set(CONFIG["cleaning"]["expected_countries"])

# -- Reproducibility: a single seed from config --
np.random.seed(CONFIG["meta"]["random_seed"])

RAW_DIR       = Path(CONFIG["paths"]["raw_dir"])
PROCESSED_DIR = Path(CONFIG["paths"]["processed_dir"])
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

colors_map = CONFIG["colors_map"]

# -- Summary of loaded parameters (transparency) --
print("\nKey parameters loaded from params.yml:")
print(f"  random_seed              : {CONFIG['meta']['random_seed']}")
print(f"  sentinel_values          : {CONFIG['cleaning']['sentinel_values']}")
print(f"  outage_hours             : {CONFIG['eda']['outage_hours']}")
print(f"  min_completeness_pct     : {CONFIG['filter']['min_completeness_pct']}")
print(f"  min_neighbors            : {CONFIG['spatial']['min_neighbors']}")
print(f"  extreme_percentile       : {CONFIG['derivation']['extreme_percentile']} (P99, not a flat 200)")
print(f"  IF contamination (NB03)  : {CONFIG['tsad']['isolation_forest_contamination']} (⚠ validated in E.8.1)")
print("\n💡 To change any parameter: edit config/params.yml, then Restart & Run All.")

✅ Configuration loaded from ../config/params.yml
   config_version: 2.0

Key parameters loaded from params.yml:
  random_seed              : 42
  sentinel_values          : [999, 999.9, 999.99, 9999, -999, -9999]
  outage_hours             : 72
  min_completeness_pct     : 50
  min_neighbors            : 3
  extreme_percentile       : 99 (P99, not a flat 200)
  IF contamination (NB03)  : 0.05 (⚠ validated in E.8.1)

💡 To change any parameter: edit config/params.yml, then Restart & Run All.


## 2. Load Raw Data

In [3]:
locations = pd.read_csv(RAW_DIR / "anomaly_locations.csv")
meas = pd.read_csv(RAW_DIR / "anomaly_measurements.csv")

# Store the RAW row counts before any cleaning (for the audit trail later)
n_raw_locations = len(locations)
n_raw_measurements = len(meas)

print("Locations shape:", locations.shape)
print("Measurements shape:", meas.shape)
print()
print("Locations columns:", locations.columns.tolist())
print("Measurements columns:", meas.columns.tolist())
print()

# Confirm that the columns we rely on are actually present
expected_loc_cols = ["location_id", "latitude", "longitude", "country"]
expected_meas_cols = ["location_id", "date", "value", "pollutant", "unit"]

missing_loc = set(expected_loc_cols) - set(locations.columns)
missing_meas = set(expected_meas_cols) - set(meas.columns)

assert not missing_loc, f"❌ Missing columns in locations: {missing_loc}"
assert not missing_meas, f"❌ Missing columns in measurements: {missing_meas}"

print("✅ All expected columns are present in both files")
print(f"   Raw locations rows   : {n_raw_locations:,}")
print(f"   Raw measurements rows: {n_raw_measurements:,}")

Locations shape: (4764, 7)
Measurements shape: (41880324, 9)

Locations columns: ['location_id', 'location_name', 'city_id', 'country', 'longitude', 'latitude', 'geometry']
Measurements columns: ['location_id', 'country_id', 'source', 'source_country', 'date', 'pollutant', 'value', 'unit', 'country']

✅ All expected columns are present in both files
   Raw locations rows   : 4,764
   Raw measurements rows: 41,880,324


---
# A. Data Cleaning — `locations`

Clean the station-location file before merging it with the measurements.

**Steps:**
1. Check structure & missing values
2. Check `location_id` duplicates
3. Validate geographic coordinates (lat/lon must fall within a valid range)
4. Standardize the `country` column

### A.1 Check Structure & Missing Values

In [4]:
print("Shape:", locations.shape)
locations.info()

Shape: (4764, 7)
<class 'pandas.DataFrame'>
RangeIndex: 4764 entries, 0 to 4763
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   location_id    4764 non-null   str    
 1   location_name  4764 non-null   str    
 2   city_id        4764 non-null   str    
 3   country        4764 non-null   str    
 4   longitude      4764 non-null   float64
 5   latitude       4764 non-null   float64
 6   geometry       4764 non-null   str    
dtypes: float64(2), str(5)
memory usage: 260.7 KB


In [5]:
missing_loc = locations.isnull().sum()
missing_loc_pct = (missing_loc / len(locations) * 100).round(2)
missing_tbl = pd.DataFrame({"missing_count": missing_loc, "missing_pct": missing_loc_pct})
print("Missing values per column (locations):")
print(missing_tbl.to_string())

# Quick summary: which columns have gaps?
cols_with_holes = missing_tbl[missing_tbl["missing_count"] > 0]
if len(cols_with_holes) == 0:
    print("\n✅ No missing values in locations")
else:
    print(f"\n⚠️  {len(cols_with_holes)} column(s) have missing values — watch these during validation")
missing_tbl

Missing values per column (locations):
               missing_count  missing_pct
location_id                0          0.0
location_name              0          0.0
city_id                    0          0.0
country                    0          0.0
longitude                  0          0.0
latitude                   0          0.0
geometry                   0          0.0

✅ No missing values in locations


,missing_count,missing_pct
location_id,0,0.0
location_name,0,0.0
city_id,0,0.0
country,0,0.0
longitude,0,0.0
latitude,0,0.0
geometry,0,0.0


### A.2 Check `location_id` Duplicates & Spurious Coordinates

In [6]:
# -- Step 1: Investigate duplicates BEFORE dropping them --
dupe_loc = locations.duplicated(subset=["location_id"]).sum()
print(f"Duplicate location_id count: {dupe_loc}")

if dupe_loc > 0:
    dup_ids = locations.loc[
        locations.duplicated(subset=["location_id"], keep=False), "location_id"
    ].unique()[:3]
    print(f"Sample duplicated location_id values (max 3): {list(dup_ids)}")
    print("Inspecting the rows to confirm keep='first' will not discard important information:")
    print(locations[locations["location_id"].isin(dup_ids)].to_string())

before = len(locations)
locations = locations.drop_duplicates(subset=["location_id"], keep="first")
after = len(locations)
print(f"\nRows removed due to duplication: {before - after}")

# -- Step 2: Detect & remove "Null Island" (0,0) coordinates --
locations["latitude"] = pd.to_numeric(locations["latitude"], errors="coerce")
locations["longitude"] = pd.to_numeric(locations["longitude"], errors="coerce")

eps = CONFIG["cleaning"]["null_island_deg"]
null_island = (locations["latitude"].abs() < eps) & (locations["longitude"].abs() < eps)
n_null_island = int(null_island.sum())
print(f"\nStations with 'Null Island' (0,0) coordinates: {n_null_island}")
if n_null_island > 0:
    print("  -> Removed: almost certainly a broken GPS reading, not an actual station location")
    locations = locations[~null_island].copy()

Duplicate location_id count: 0

Rows removed due to duplication: 0

Stations with 'Null Island' (0,0) coordinates: 0


### A.3 Validate Geographic Coordinates

Latitude must fall within [-90, 90] and longitude within [-180, 180].

In [7]:
locations["latitude"] = pd.to_numeric(locations["latitude"], errors="coerce")
locations["longitude"] = pd.to_numeric(locations["longitude"], errors="coerce")

invalid_coords_loc = (
    (locations["latitude"] < -90) | (locations["latitude"] > 90) |
    (locations["longitude"] < -180) | (locations["longitude"] > 180) |
    locations["latitude"].isna() | locations["longitude"].isna()
)

print(f"Stations with invalid coordinates: {invalid_coords_loc.sum()}")

before = len(locations)
locations = locations[~invalid_coords_loc].copy()
after = len(locations)
print(f"Stations removed: {before - after}")

Stations with invalid coordinates: 0
Stations removed: 0


### A.4 Standardize the `country` Column

`.str.title()` is applied first for initial case normalization, after which an
explicit mapping unifies every spelling variant of USA (`"Usa"`, `"United States"`,
`"US"`, etc.) into a single canonical value, `"USA"`.
Without this mapping, joins across files that use different spellings could fail silently.

In [8]:
locations["country"] = locations["country"].str.strip().str.title()

# Explicit mapping: unify all USA spelling variants -> "USA"
_country_map = {
    "Usa": "USA",
    "Us": "USA",
    "U.S.": "USA",
    "United States": "USA",
    "United States Of America": "USA",
}
locations["country"] = locations["country"].replace(_country_map)

print("Unique countries in locations:")
print(locations["country"].value_counts())

Unique countries in locations:
country
China      1931
USA        1415
Germany     878
India       540
Name: count, dtype: int64


In [9]:
print("=== Locations after cleaning ===")
print("Shape:", locations.shape)
locations.head()

=== Locations after cleaning ===
Shape: (4764, 7)


,location_id,location_name,city_id,country,longitude,latitude,geometry
0,1410a,海南大学,haikou_chn.9_1_cn,China,110.3232,20.0574,"c(110.3232, 20.0574)"
1,1450a,呈贡新区,kunming_chn.30_1_cn,China,102.8204,24.8886,"c(102.8204, 24.8886)"
2,1451a,西山森林公园(对照点),kunming_chn.30_1_cn,China,102.6251,24.9608,"c(102.6251, 24.9608)"
3,1452a,龙泉镇,kunming_chn.30_1_cn,China,102.7252,25.0902,"c(102.7252, 25.0902)"
4,1455a,碧鸡广场,kunming_chn.30_1_cn,China,102.6644,25.0401,"c(102.6644, 25.0401)"


---
# B. Data Cleaning — `measurements`

Clean the PM2.5 measurement file before merging it with locations.

**Steps:**
1. Check structure & missing values
2. Drop 100%-null columns
3. Validate pollutant & unit
4. Check duplicates (`location_id` + `date`)
5. Handle sentinel values (sensor error codes)
6. Handle negative values (set to `NaN`)
7. Convert the `date` column data type
8. Save the cleaning log

### B.1 Check Structure & Missing Values

We examine whether missingness correlates with a particular time or station BEFORE discarding it, because the pattern itself may be the anomaly signal we are looking for.

In [10]:
print("Shape:", meas.shape)
meas.info()

Shape: (41880324, 9)
<class 'pandas.DataFrame'>
RangeIndex: 41880324 entries, 0 to 41880323
Data columns (total 9 columns):
 #   Column          Dtype  
---  ------          -----  
 0   location_id     str    
 1   country_id      str    
 2   source          str    
 3   source_country  str    
 4   date            str    
 5   pollutant       str    
 6   value           float64
 7   unit            str    
 8   country         float64
dtypes: float64(2), str(7)
memory usage: 2.8 GB


In [11]:
missing_meas = meas.isnull().sum()
missing_meas_pct = (missing_meas / len(meas) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing_meas, "missing_pct": missing_meas_pct})
print("Missing values per column (measurements):")
print(missing_summary[missing_summary["missing_count"] > 0].to_string())
missing_summary[missing_summary["missing_count"] > 0]

Missing values per column (measurements):
         missing_count  missing_pct
country       41880324        100.0


,missing_count,missing_pct
country,41880324,100.0


In [12]:
# -- Examine the PATTERN of missingness in 'value' before dropping anything --
# Parse the date to test whether the missingness follows an hourly pattern
_tmp = meas.copy()
_tmp["_date_parsed"] = pd.to_datetime(_tmp["date"], utc=True, errors="coerce")
_valid_date = _tmp["_date_parsed"].notna()

if _valid_date.any():
    _tmp_v = _tmp[_valid_date].copy()
    _tmp_v["_hour"] = _tmp_v["_date_parsed"].dt.hour
    missing_by_hour = _tmp_v.groupby("_hour")["value"].apply(lambda x: x.isna().mean() * 100)
    print("Percentage of missing 'value' entries per hour (0-23):")
    print("  (roughly flat -> random; some hours standing out -> systematic pattern)")
    print(missing_by_hour.round(1).to_string())
    _spread = missing_by_hour.max() - missing_by_hour.min()
    print(f"\n  Spread between the highest and lowest hour: {_spread:.1f} percentage points")
    if _spread > 15:
        print("  ⚠️  There is an indication of a temporal PATTERN in the missingness — record as a finding.")
    else:
        print("  ✅ Missingness is fairly even across hours — likely random.")

del _tmp

Percentage of missing 'value' entries per hour (0-23):
  (roughly flat -> random; some hours standing out -> systematic pattern)
_hour
0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
20    0.0
21    0.0
22    0.0
23    0.0

  Spread between the highest and lowest hour: 0.0 percentage points
  ✅ Missingness is fairly even across hours — likely random.


In [13]:
# -- Now drop rows with missing values in the critical columns --
critical_cols = ["location_id", "date", "value"]
before = len(meas)
meas = meas.dropna(subset=critical_cols)
after = len(meas)
n_dropped_critical = before - after
print(f"Rows dropped due to missing values in critical columns {critical_cols}: {n_dropped_critical:,}")
print(f"  ({n_dropped_critical / before * 100:.2f}% of the data before this step)")

Rows dropped due to missing values in critical columns ['location_id', 'date', 'value']: 0
  (0.00% of the data before this step)


### B.2 Drop 100%-Null Columns

The `country` column in the measurements file is 100% empty. We drop it **before** the merge so that it does not later collide with the genuine (populated) `country` column from the locations file. If the two `country` columns meet during the merge, pandas will rename them `country_x` and `country_y` — a source of confusion and bugs.

In [14]:
null_cols = [col for col in meas.columns if meas[col].isna().all()]
print(f"100%-null columns dropped: {null_cols}")
meas = meas.drop(columns=null_cols)
print(f"Remaining columns: {meas.columns.tolist()}")

100%-null columns dropped: ['country']
Remaining columns: ['location_id', 'country_id', 'source', 'source_country', 'date', 'pollutant', 'value', 'unit']


### B.3 Validate Pollutant & Unit

We ensure that **all** data are PM2.5 expressed in µg/m³. Two spellings of the unit are accepted: `µg/m³` (with a superscript) and `µg/m3` (plain).

In [15]:
print(f"Unique pollutants before filtering: {meas['pollutant'].value_counts().to_dict()}")
before_poll = len(meas)
meas = meas[meas["pollutant"] == "pm25"].copy()
n_dropped_pollutant = before_poll - len(meas)
print(f"Non-pm25 rows dropped: {n_dropped_pollutant:,}")

print(f"\nUnique units before filtering: {meas['unit'].value_counts().to_dict()}")
valid_units = ["µg/m³", "µg/m3"]
before_unit = len(meas)
meas = meas[meas["unit"].isin(valid_units)].copy()
n_dropped_unit = before_unit - len(meas)
print(f"Rows with invalid units dropped: {n_dropped_unit:,}")

print(f"\nRemaining units    : {meas['unit'].unique().tolist()}")
print(f"Remaining pollutants: {meas['pollutant'].unique().tolist()}")

Unique pollutants before filtering: {'pm25': 41880324}
Non-pm25 rows dropped: 0

Unique units before filtering: {'µg/m3': 41880324}
Rows with invalid units dropped: 0

Remaining units    : ['µg/m3']
Remaining pollutants: ['pm25']


### B.4 Check and Remove Duplicates (`location_id` + `date`)

We check whether any station reports duplicate measurements, and whether such duplicates hold identical or conflicting values, before removing them.

In [16]:
dupes = meas.duplicated().sum()
print(f"Number of FULLY duplicate rows (all columns identical): {dupes:,}")

dupes_key = meas.duplicated(subset=["location_id", "date"]).sum()
print(f"Duplicates on the location_id + date combination: {dupes_key:,}")

# -- Investigate: do the duplicate keys carry different values? --
if dupes_key > 0:
    dup_mask = meas.duplicated(subset=["location_id", "date"], keep=False)
    sample = meas[dup_mask].sort_values(["location_id", "date"]).head(6)
    print("\nSample duplicate rows (location_id + date) for investigation:")
    print(sample[["location_id", "date", "value"]].to_string(index=False))

    # Check whether the values within these duplicates are identical or different
    grp = meas[dup_mask].groupby(["location_id", "date"])["value"].nunique()
    n_conflicting = int((grp > 1).sum())
    if n_conflicting > 0:
        print(f"\n⚠️  {n_conflicting} duplicate pair(s) carry DIFFERENT VALUES —")
        print("   this is not merely a download artifact. Record it as a potential finding.")
    else:
        print("\n✅ All duplicates carry identical values — likely a download artifact, safe to drop.")

Number of FULLY duplicate rows (all columns identical): 0
Duplicates on the location_id + date combination: 0


In [17]:
before = len(meas)
meas = meas.drop_duplicates()
meas = meas.drop_duplicates(subset=["location_id", "date"], keep="first")
after = len(meas)
print(f"Rows removed due to duplication: {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows removed due to duplication: 0
Rows remaining: 41,880,324


### B.5 Sentinel Values → NaN

Values such as `999`, `999.9`, `999.99`, and `9999` are common sensor error codes — not genuine measurements.

Genuinely high PM2.5 values (wildfires, severe smog) are **not** converted to NaN here
— TSAD will assess contextually whether such values are anomalous.

In [18]:
sentinel_mask = meas["value"].isin(CONFIG["cleaning"]["sentinel_values"])
n_sentinel = int(sentinel_mask.sum())
meas.loc[sentinel_mask, "value"] = np.nan

print(f"Sentinel values {CONFIG['cleaning']['sentinel_values']} -> NaN: {n_sentinel}")
print(f"Other extreme values were left untouched — TSAD will assess them contextually")

Sentinel values [999, 999.9, 999.99, 9999, -999, -9999] -> NaN: 43
Other extreme values were left untouched — TSAD will assess them contextually


### B.6 Negative Values → NaN

A negative PM2.5 reading is physically impossible — it is almost certainly sensor noise or a calibration error.
A value of 0 µg/m³ is a valid, genuine reading (extremely clean air), so negative values are set to
`NaN`, **not clipped to 0**, so as not to distort the station's reference distribution.

In [19]:
n_negative = int((meas["value"] < 0).sum())
print(f"Negative values found: {n_negative} ({n_negative / len(meas) * 100:.4f}%)")

meas.loc[meas["value"] < 0, "value"] = np.nan

print(f"After cleaning — missing values in the 'value' column: {meas['value'].isna().sum()}")
print(f"Value range now: {meas['value'].min():.1f} - {meas['value'].max():.1f} µg/m³")

Negative values found: 296203 (0.7073%)
After cleaning — missing values in the 'value' column: 296246
Value range now: 0.0 - 2158.0 µg/m³


### B.7 Convert Date to UTC Datetime

In [20]:
meas["date"] = pd.to_datetime(meas["date"], utc=True, errors="coerce")

n_invalid_dates = int(meas["date"].isna().sum())
print(f"Invalid dates (NaN after conversion): {n_invalid_dates}")
if n_invalid_dates > 0:
    meas = meas.dropna(subset=["date"])
    print(f"Rows removed: {n_invalid_dates}")

print(f"Date range: {meas['date'].min()} to {meas['date'].max()}")
print(f"Date column dtype: {meas['date'].dtype}")

Invalid dates (NaN after conversion): 0
Date range: 2021-01-01 00:00:00+00:00 to 2023-12-31 23:00:00+00:00
Date column dtype: datetime64[us, UTC]


In [21]:
print("=== Measurements after cleaning ===")
print("Shape:", meas.shape)
meas.head()

=== Measurements after cleaning ===
Shape: (41880324, 8)


,location_id,country_id,source,source_country,date,pollutant,value,unit
0,1001a,CN,mee,China,2023-01-01 00:00:00+00:00,pm25,16.0,µg/m3
1,1001a,CN,mee,China,2023-01-01 01:00:00+00:00,pm25,16.0,µg/m3
2,1001a,CN,mee,China,2023-01-01 02:00:00+00:00,pm25,15.0,µg/m3
3,1001a,CN,mee,China,2023-01-01 03:00:00+00:00,pm25,12.0,µg/m3
4,1001a,CN,mee,China,2023-01-01 04:00:00+00:00,pm25,11.0,µg/m3


### B.8 Cleaning Log

Save a summary of every cleaning step to `cleaning_log.json` in `PROCESSED_DIR`
for reproducibility and audit purposes.

In [22]:
import json
from datetime import datetime, timezone

cleaning_log = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "B1_missing_critical": {"rows_dropped": n_dropped_critical},
    "B2_drop_null_cols": {"cols_dropped": null_cols},
    "B3_pollutant_unit_filter": {
        "rows_dropped_pollutant": n_dropped_pollutant,
        "rows_dropped_unit": n_dropped_unit,
    },
    "B4_duplicates": {"key_dupes_dropped": int(dupes_key)},
    "B5_sentinel_values": {"n_set_nan": n_sentinel},
    "B6_negative_values": {"n_set_nan": n_negative},
    "B7_date_conversion": {"invalid_dates_dropped": n_invalid_dates},
    "final_shape": list(meas.shape),
    "metadata": {
        "n_raw_locations": int(n_raw_locations),
        "n_raw_measurements": int(n_raw_measurements),
        "notebook_version": "01_v2.0_supervised",
        "utc_assumption": "OpenAQ data is assumed to be UTC — verify against the API documentation",
    },
}

log_path = PROCESSED_DIR / "cleaning_log.json"
with open(log_path, "w") as f:
    json.dump(cleaning_log, f, indent=2, default=str)

print(f"Cleaning log saved to: {log_path}")
print(json.dumps(cleaning_log, indent=2, default=str))

Cleaning log saved to: ../data/processed/cleaning_log.json
{
  "timestamp": "2026-07-07T07:49:49.091314+00:00",
  "B1_missing_critical": {
    "rows_dropped": 0
  },
  "B2_drop_null_cols": {
    "cols_dropped": [
      "country"
    ]
  },
  "B3_pollutant_unit_filter": {
    "rows_dropped_pollutant": 0,
    "rows_dropped_unit": 0
  },
  "B4_duplicates": {
    "key_dupes_dropped": 0
  },
  "B5_sentinel_values": {
    "n_set_nan": 43
  },
  "B6_negative_values": {
    "n_set_nan": 296203
  },
  "B7_date_conversion": {
    "invalid_dates_dropped": 0
  },
  "final_shape": [
    41880324,
    8
  ],
  "metadata": {
    "n_raw_locations": 4764,
    "n_raw_measurements": 41880324,
    "notebook_version": "01_v2.0_supervised",
    "utc_assumption": "OpenAQ data is assumed to be UTC \u2014 verify against the API documentation"
  }
}


---
# C. Merge `locations` + `measurements` (Cleaned)

The two independently cleaned datasets are now joined on the `location_id` column.
`country` is taken from `locations` only — the `country` column in measurements was
already dropped in B.2.

In [23]:
loc_clean = locations[["location_id", "country", "longitude", "latitude"]].copy()

df = meas.merge(loc_clean, on="location_id", how="left")

print("Shape after merge:", df.shape)
print("Columns:", df.columns.tolist())

missing_country = df["country"].isna().sum()
print(f"\nRows without a country (no match in locations): {missing_country:,}")
df.head()

Shape after merge: (41880324, 11)
Columns: ['location_id', 'country_id', 'source', 'source_country', 'date', 'pollutant', 'value', 'unit', 'country', 'longitude', 'latitude']

Rows without a country (no match in locations): 0


,location_id,country_id,source,source_country,date,pollutant,value,unit,country,longitude,latitude
0,1001a,CN,mee,China,2023-01-01 00:00:00+00:00,pm25,16.0,µg/m3,China,116.3621,39.8784
1,1001a,CN,mee,China,2023-01-01 01:00:00+00:00,pm25,16.0,µg/m3,China,116.3621,39.8784
2,1001a,CN,mee,China,2023-01-01 02:00:00+00:00,pm25,15.0,µg/m3,China,116.3621,39.8784
3,1001a,CN,mee,China,2023-01-01 03:00:00+00:00,pm25,12.0,µg/m3,China,116.3621,39.8784
4,1001a,CN,mee,China,2023-01-01 04:00:00+00:00,pm25,11.0,µg/m3,China,116.3621,39.8784


We verify the percentage of records that were successfully joined. The accepted match rate is above 98%.

In [24]:
# -- Merge quality check: what percentage was successfully joined? --
n_before_merge = len(df)
matched = int(df["country"].notna().sum())
unmatched = int(df["country"].isna().sum())
match_rate = round(matched / n_before_merge * 100, 2)

merge_quality = {
    "total_meas_before_merge": n_before_merge,
    "matched": matched,
    "unmatched": unmatched,
    "match_rate_pct": match_rate,
}
print("=== Merge Quality ===")
for k, v in merge_quality.items():
    print(f"  {k:28s}: {v:,}" if isinstance(v, int) else f"  {k:28s}: {v}")

if match_rate < 98:
    print(f"\n⚠️  Match rate {match_rate}% < 98% — investigate the location_id values that failed to match.")
    print("   Possible cause: IDs in measurements that do not exist in locations (inconsistent source data).")
else:
    print(f"\n✅ Match rate {match_rate}% — the join is healthy.")

# -- Drop rows that failed to merge --
before = len(df)
df = df.dropna(subset=["country"])
after = len(df)
print(f"\nRows removed (failed to join to locations): {before - after:,}")

print("\nCountries after the merge:")
print(df["country"].value_counts().to_string())

=== Merge Quality ===
  total_meas_before_merge     : 41,880,324
  matched                     : 41,880,324
  unmatched                   : 0
  match_rate_pct              : 100.0

✅ Match rate 100.0% — the join is healthy.

Rows removed (failed to join to locations): 0

Countries after the merge:
country
China      13782736
USA        12872578
India       8191659
Germany     7033351


In [25]:
output_path = PROCESSED_DIR / "anomaly_merged.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path.resolve()}")
print(f"Total rows: {len(df):,} | Total stations: {df['location_id'].nunique():,}")

Saved: /Users/endarlani/Documents/spatiotemporal-air-quality-anomaly-detection/data/processed/anomaly_merged.csv
Total rows: 41,880,324 | Total stations: 3,328


---
# D. Cleaning Summary & Descriptive Statistics

In [26]:
print("="*50)
print("DATA CLEANING SUMMARY")
print("="*50)
print(f"Final row count             : {len(df):,}")
print(f"Unique stations             : {df['location_id'].nunique():,}")
print(f"Date range                  : {df['date'].min()} to {df['date'].max()}")
print(f"Missing values ('value' col): {df['value'].isna().sum():,}")
print(f"PM2.5 value range           : {df['value'].min():.1f} - {df['value'].max():.1f} µg/m³")
print(f"Countries                   : {df['country'].unique().tolist()}")

DATA CLEANING SUMMARY
Final row count             : 41,880,324
Unique stations             : 3,328
Date range                  : 2021-01-01 00:00:00+00:00 to 2023-12-31 23:00:00+00:00
Missing values ('value' col): 296,246
PM2.5 value range           : 0.0 - 2158.0 µg/m³
Countries                   : ['China', 'Germany', 'India', 'USA']


In [27]:
print("df.describe() — a rough overview (caution: the mean is misleading for skewed data):")
df.describe()

df.describe() — a rough overview (caution: the mean is misleading for skewed data):


,value,longitude,latitude
count,4.158408e+07,4.188032e+07,4.188032e+07
mean,2.651534e+01,2.397936e+01,3.600582e+01
std,3.905394e+01,8.997951e+01,1.040138e+01
min,0.000000e+00,-1.580886e+02,8.514909e+00
25%,6.126500e+00,-8.039504e+01,2.855120e+01
50%,1.320000e+01,7.363214e+01,3.571970e+01
75%,3.100000e+01,1.095680e+02,4.429210e+01
max,2.158000e+03,1.311652e+02,6.484593e+01


**PM2.5 data are NOT symmetric bell-shaped (normally distributed).** They are skewed — mostly low values, with occasional very high values (during fires or smog). This shape is termed *log-normal*.

**Why this matters:** the `mean` and `std` from `df.describe()` are **misleading** for skewed data. The mean is "pulled" by extreme values. More honest summaries are the **median** and the **IQR** (interquartile range).

We complement `describe()` with the median, IQR, skewness (a measure of asymmetry), and a comparison against the WHO guidelines — so that the picture of the data is honest.

In [ ]:
# -- More honest statistics for PM2.5 data (log-normal) --
from scipy import stats as _scipy_stats

WHO_ANNUAL = CONFIG["derivation"]["who_annual_ugm3"]   # WHO 2021 annual mean guideline
WHO_24H    = CONFIG["derivation"]["who_24h_ugm3"]       # WHO 2021 24-hour guideline

print("="*72)
print("PER-COUNTRY STATISTICS (median & IQR are more honest than the mean for PM2.5)")
print("="*72)
print(f"{'Country':<10} {'Median':>8} {'IQR':>8} {'Skew':>7} {'>WHO_yr(5)':>11} {'>WHO_24h(15)':>13}")
print("-"*72)
for country in sorted(df["country"].dropna().unique()):
    vals = df.loc[df["country"] == country, "value"].dropna()
    if len(vals) == 0:
        continue
    median = vals.median()
    iqr = vals.quantile(0.75) - vals.quantile(0.25)
    skew = _scipy_stats.skew(vals)
    pct_over_annual = (vals > WHO_ANNUAL).mean() * 100
    pct_over_24h = (vals > WHO_24H).mean() * 100
    print(f"{country:<10} {median:>8.1f} {iqr:>8.1f} {skew:>7.2f} "
          f"{pct_over_annual:>10.0f}% {pct_over_24h:>12.0f}%")

print("-"*72)
print("Note: skew > 0 = right-skewed (many low values, a few very high ones).")
print("      This is NORMAL and EXPECTED for PM2.5 data (Ott 1990).")
print("      The >WHO columns show how often air quality exceeds the health guidelines.")

PER-COUNTRY STATISTICS (median & IQR are more honest than the mean for PM2.5)
Country      Median      IQR    Skew  >WHO_yr(5)  >WHO_24h(15)
------------------------------------------------------------------------
China          24.0     27.0    4.25         94%           70%
Germany         7.3      7.7    6.61         68%           16%
India          39.5     50.6    3.41         98%           85%
USA             6.3      6.0   16.96         61%           10%
------------------------------------------------------------------------
Note: skew > 0 = right-skewed (many low values, a few very high ones).
      This is NORMAL and EXPECTED for PM2.5 data (Ott 1990).
      The >WHO columns show how often air quality exceeds the health guidelines.


---
## ✅ End of Notebook 01

**Outputs produced by this notebook:**
- `data/processed/anomaly_merged.csv` — the cleaned, merged dataset
- `data/processed/cleaning_log.json` — an audit trail of every cleaning step

In [29]:
# -- Final verification: confirm the merged file exists --
_merged = PROCESSED_DIR / "anomaly_merged.csv"
assert _merged.exists(), f"❌ {_merged} has not been created — run Section C first."
print(f"✅ Handoff file ready for NB02: {_merged}")
print(f"   Size: {_merged.stat().st_size / 1e6:.1f} MB")
print(f"   Rows : {len(df):,} | Stations: {df['location_id'].nunique():,}")
print()
print("➡️  Continue to: 02_exploratory_data_analysis.ipynb")

✅ Handoff file ready for NB02: ../data/processed/anomaly_merged.csv
   Size: 4052.8 MB
   Rows : 41,880,324 | Stations: 3,328

➡️  Continue to: 02_exploratory_data_analysis.ipynb
